# Data preparation des données de data.gouv - liste des communes
### sans polygones
data.gouv propose un jeu en geojson avec polygones (voir README.md)

In [15]:
import sys
from datetime import datetime, timedelta
from pathlib import Path

import chardet
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import seaborn as sns
from folium.plugins import HeatMap, MarkerCluster

# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import GEO_DATA_RAW_DIR, GEO_DATA_CLEAN_DIR
from utils.cleaning_utils import delete, normalize_columns_names, normalize_text_columns_cells, optimize_numeric_column, fill_rate
from utils.analysis_utils import plot_missing_bar, plot_numeric_histograms, plot_corr_heatmap, plot_qualitative


In [16]:
pd.set_option('display.max_columns', None)

In [17]:
RAW_DATA_FILE = GEO_DATA_RAW_DIR / 'communes-france-2026.csv'

RAW_DATA_FILE

PosixPath('/Users/brunocoulet/Documents/projets/incendies/data/geo_data_raw/communes-france-2026.csv')

In [18]:
# Check the encoding of the CSV files
with open(RAW_DATA_FILE, 'rb') as file:
    encodage = chardet.detect(file.read(10000))

print(encodage)

{'encoding': 'utf-8', 'confidence': 0.832832, 'language': 'fr', 'mime_type': 'text/plain'}


In [19]:
df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')

/var/folders/5b/52dg249x1sv6tfc69m3y88z80000gq/T/ipykernel_55236/650886494.py:1: DtypeWarning: Columns (0: code_insee, 1: dep_code, 2: canton_code, 3: epci_code, 4: code_insee_centre_zone_emploi, 5: code_unite_urbaine, 6: code_insee_centre_aire_attraction, 7: code_bassin_de_vie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34868 entries, 0 to 34867
Data columns (total 62 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   code_insee                             34868 non-null  object 
 1   nom_standard                           34868 non-null  string 
 2   nom_sans_pronom                        34868 non-null  string 
 3   nom_a                                  34868 non-null  string 
 4   nom_de                                 34868 non-null  string 
 5   nom_sans_accent                        34868 non-null  string 
 6   nom_standard_majuscule                 34868 non-null  string 
 7   typecom                                34868 non-null  string 
 8   typecom_texte                          34868 non-null  string 
 9   reg_code                               34868 non-null  Int64  
 10  reg_nom                                34868 non-null  string 
 11  dep_code     

In [21]:
df.columns

Index(['code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de',
       'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte',
       'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code',
       'canton_nom', 'epci_code', 'epci_nom', 'code_postal', 'codes_postaux',
       'academie_code', 'academie_nom', 'zone_emploi', 'zone_emploi_nom',
       'code_insee_centre_zone_emploi', 'code_unite_urbaine',
       'nom_unite_urbaine', 'taille_unite_urbaine',
       'type_commune_unite_urbaine', 'statut_commune_unite_urbaine',
       'code_aire_attraction', 'nom_aire_attraction',
       'categorie_aire_attraction', 'taille_aire_attraction',
       'code_insee_centre_aire_attraction', 'code_bassin_de_vie',
       'nom_bassin_de_vie', 'type_bassin_de_vie', 'population',
       'superficie_hectare', 'superficie_km2', 'densite',
       'superficie_cartographique_ign_hectare',
       'superficie_cartographique_ign_km2', 'densite_cartographique',
       'superficie_c

### Sélection/renommage/typage des variables
- altitude
'altitude_centre' à des NaN, on le supprime<br>
'altitude_mairie'  devient 'altitude'<br>

- code_postal<br>
    renommé 'localisation'

- region<br>
    'reg_code' devien: 'region'

- departement<br>
    'dep_code' devient 'departement'

In [22]:
# Dictionnaire de correspondance (Ancien nom -> Nouveau nom pour tes analyses)
colonnes_mapping = {
    'code_insee': 'code_insee',
    'code_postal': 'localisation',       # Correspond au code postal principal
    'nom_standard': 'nom_standard',
    'reg_code': 'region',                # Renommé pour l'analyse
    'dep_code': 'departement',           # Renommé pour l'analyse
    'population': 'population',
    'superficie_hectare': 'superficie_hectare',
    'densite': 'densite',
    'altitude_moyenne': 'altitude_moyenne',
    'altitude_minimale': 'altitude_minimale',
    'altitude_maximale': 'altitude_maximale',
    'latitude_mairie': 'latitude',
    'longitude_mairie': 'longitude',
    'grille_densite' : 'grille_densite'
}

# filtre le DataFrame d'origine pour ne garder que ces colonnes et les renommer
df_filtre = df[list(colonnes_mapping.keys())].rename(columns=colonnes_mapping)

# Définit les types cibles compatibles avec PostgreSQL
# Utiliser des majuscules (Int32, Float64, string) permet à Pandas de gérer les valeurs absentes (NaN) sans bloquer
types_cibles = {
    'code_insee': 'string',
    'localisation': 'Int32',
    'nom_standard': 'string',
    'region': 'Int8',
    'departement': 'string',  # En string pour garder les préfixes ou formats spéciaux si besoin
    'population': 'Int32',
    'superficie_hectare': 'Int32',
    'densite': 'Float64',
    'altitude_moyenne': 'Int16',
    'altitude_minimale': 'Int16',
    'altitude_maximale': 'Int16',
    'latitude': 'Float64',
    'longitude': 'Float64',
    'grille_densite': 'Int8'
}

# applique le typage au DataFrame
df = df_filtre.astype(types_cibles)

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34868 entries, 0 to 34867
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_insee          34868 non-null  string 
 1   localisation        34868 non-null  Int32  
 2   nom_standard        34868 non-null  string 
 3   region              34868 non-null  Int8   
 4   departement         34868 non-null  string 
 5   population          34868 non-null  Int32  
 6   superficie_hectare  34739 non-null  Int32  
 7   densite             34739 non-null  Float64
 8   altitude_moyenne    34863 non-null  Int16  
 9   altitude_minimale   34863 non-null  Int16  
 10  altitude_maximale   34863 non-null  Int16  
 11  latitude            34863 non-null  Float64
 12  longitude           34863 non-null  Float64
 13  grille_densite      34868 non-null  Int8   
dtypes: Float64(3), Int16(3), Int32(3), Int8(2), string(3)
memory usage: 3.3 MB


In [24]:
df.head()

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude,longitude,grille_densite
0,01001,1400,L'Abergement-Clémenciat,84,01,860,1562,55.060001,242,206,272,46.151,4.921,6
1,01002,1640,L'Abergement-de-Varey,84,01,270,918,29.43,483,290,748,46.007,5.423,6
2,01004,1500,Ambérieu-en-Bugey,84,01,15934,2451,650.130005,379,237,753,45.958,5.36,2
3,01005,1330,Ambérieux-en-Dombes,84,01,1906,1601,119.019997,290,265,302,45.996,4.903,5
4,01006,1300,Ambléon,84,01,115,603,19.07,589,330,940,45.748,5.601,6


In [25]:
df.shape

(34868, 14)

### Métropole, Corse, Outre Mer - Filtre outre_mer
- Corse: 2A, 2B
- Outre-mer : 971 à 976

In [26]:
# Exclut les départements d'outre-mer
corse = ['2A', '2B']
outre_mer = ['971', '972', '973', '974', '976', '977', '978', '984', '986', '987', '988']

df = df[~df['departement'].isin(outre_mer)]

In [27]:
df.shape

(34739, 14)

## Export

In [28]:
df.to_parquet(
    GEO_DATA_CLEAN_DIR / "communes_metropole_corse.parquet",
    index=False
)